In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils import data
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
lr, epoch, batch_size = 0.05, 10, 256

In [2]:
def data_maker():
    #transform = transforms.ToTensor()
    
    transform = transforms.Compose([
        transforms.Resize((96, 96)),
        transforms.ToTensor()
    ])
    

    train_dataset = datasets.FashionMNIST(
        root='../data',
        train=True,
        transform=transform,
        download=True
    )

    test_dataset = datasets.FashionMNIST(
        root='../data',
        train=False,
        transform=transform,
        download=True
    )
    return train_dataset,test_dataset

In [3]:
def data_loader(data_array,batch_size,flag):
    return data.DataLoader(data_array,batch_size,shuffle=flag)


In [4]:
class residual_normal(nn.Module):
    def __init__(self,in_channel,out_channel):
        super().__init__()
        self.net=nn.Sequential()
        self.net.append(nn.Conv2d(in_channel,out_channel,kernel_size=3,stride=1,padding=1))
        self.net.append(nn.BatchNorm2d(out_channel))
        self.net.append(nn.ReLU())
        self.net.append(nn.Conv2d(out_channel,out_channel,kernel_size=3,stride=1,padding=1))
        self.net.append(nn.BatchNorm2d(out_channel))
    def forward(self,x):
        y=self.net(x)
        y=y+x
        return F.relu(y)
    
class residual(nn.Module):
    def __init__(self,in_channel,out_channel):
        super().__init__()
        self.net=nn.Sequential()
        self.net.append(nn.Conv2d(in_channel,out_channel,kernel_size=3,stride=2,padding=1))
        self.net.append(nn.BatchNorm2d(out_channel))
        self.net.append(nn.ReLU())
        self.net.append(nn.Conv2d(out_channel,out_channel,kernel_size=3,stride=1,padding=1))
        self.net.append(nn.BatchNorm2d(out_channel))
        self.net1=nn.Conv2d(in_channel,out_channel,kernel_size=1,stride=2)
    def forward(self,x):
        y=self.net(x)+self.net1(x)
        return F.relu(y)


In [5]:
b1 = nn.Sequential(nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3),
                   nn.BatchNorm2d(64), nn.ReLU(),
                   nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
b2=nn.Sequential(residual_normal(64,64),residual_normal(64,64))
b3=nn.Sequential(residual(64,128),residual_normal(128,128))
b4=nn.Sequential(residual(128,256),residual_normal(256,256))
b5=nn.Sequential(residual(256,512),residual_normal(512,512))
net=nn.Sequential(b1,b2,b3,b4,b5,nn.AdaptiveAvgPool2d((1,1)),nn.Flatten(),nn.Linear(512,10))

In [6]:

def init(m):
    if isinstance(m,nn.Linear) or isinstance(m,nn.Conv2d):
        nn.init.xavier_uniform_(m.weight)

In [7]:
train_dataset,test_dataset=data_maker()
train_iter=data_loader(train_dataset,batch_size,True)
test_iter=data_loader(test_dataset,batch_size,False)
net.apply(init)
net=net.to('cuda')
loss=nn.CrossEntropyLoss()

trainer=torch.optim.SGD(net.parameters(),lr=lr)


In [8]:
for i in range(epoch):
    net.train()
    for x,y in train_iter:
        x=x.to("cuda")
        y=y.to("cuda")
        trainer.zero_grad()
        l=loss(net(x),y)
        l.backward()
        trainer.step()
    net.eval()
    train_acc=0
    test_acc=0
    train_loss=0.0
    test_loss=0.0
    train_count=0
    test_count=0
    with torch.no_grad():
        for x,y in train_iter:
            x=x.to("cuda")
            y=y.to("cuda")
            logit=net(x)
            train_count+=y.size(0)
            l=loss(logit,y)
            train_loss+=l.item()*y.size(0)
            train_acc+=(logit.argmax(dim=1)==y).sum().item()
        print(f"第{i+1}轮的训练loss：{train_loss/train_count:.5f},训练准确率：{train_acc/train_count}")
        for x,y in test_iter:
            x=x.to("cuda")
            y=y.to("cuda")
            logit=net(x)
            test_count+=y.size(0)
            l=loss(logit,y)
            test_loss+=l.item()*y.size(0)
            test_acc+=(logit.argmax(dim=1)==y).sum().item()
        print(f"第{i+1}轮的测试loss：{test_loss/test_count:.5f},测试准确率：{test_acc/test_count}")



第1轮的训练loss：0.39703,训练准确率：0.8515833333333334
第1轮的测试loss：0.45755,测试准确率：0.832
第2轮的训练loss：0.25331,训练准确率：0.90735
第2轮的测试loss：0.33103,测试准确率：0.8806
第3轮的训练loss：0.27652,训练准确率：0.8948666666666667
第3轮的测试loss：0.37217,测试准确率：0.8617
第4轮的训练loss：0.24093,训练准确率：0.9140333333333334
第4轮的测试loss：0.37717,测试准确率：0.8777
第5轮的训练loss：0.18469,训练准确率：0.9336
第5轮的测试loss：0.36055,测试准确率：0.8801
第6轮的训练loss：0.37042,训练准确率：0.8830833333333333
第6轮的测试loss：0.55660,测试准确率：0.8368
第7轮的训练loss：0.06858,训练准确率：0.9765166666666667
第7轮的测试loss：0.33199,测试准确率：0.9075
第8轮的训练loss：0.16013,训练准确率：0.9430666666666667
第8轮的测试loss：0.44191,测试准确率：0.8824
第9轮的训练loss：0.28356,训练准确率：0.9130666666666667
第9轮的测试loss：0.51755,测试准确率：0.8637
第10轮的训练loss：0.03292,训练准确率：0.98835
第10轮的测试loss：0.33158,测试准确率：0.91
